# ALBEF 模型结构与创新点笔记

## 一、背景与动机
ALBEF（A Learned Representation for Embodied Agents）是在2021年NIPS会议上发布的一项研究，旨在改进现有的视觉-语言预训练模型。在此之前，大多数视觉预训练模型依赖于目标检测器（detector）来提取图像特征，这导致了以下问题：
- 计算复杂度高：目标检测器包含大量的anchor和bounding box，增加了计算负担。
- 数据限制：目标检测器通常在标注数据集（如COCO）上训练，这些数据集中的物体类别有限，限制了模型的表现。

## 二、模型结构

### 1. 编码器设计
ALBEF摒弃了传统的基于检测器的方法，采用了更高效的编码器架构：
- 图像编码器：使用12层的Vision Transformer (ViT) Base模型，具有16个head的Transformer。
- 文本编码器：采用BERT模型，分为前后两部分，前6层用于编码文本，后6层用于多模态融合。

### 2. 特征交互与损失
- 全局表示：通过自注意力机制，图像和文本的CLS token嵌入包含了全局信息，可用于构建对比学习的损失函数。
- 多模态融合：图像和文本的embedding通过cross-attention机制在多模态编码器中进一步融合，支持两个主要任务：
    - 图像-文本匹配（ITM）：判断图像和文本是否匹配，除了有当前样本的二分类损失，还有hard negatives的二分类损失
    - 掩码语言建模（MLM）：根据上下文恢复被mask的单词，利用多模态信息增强融合效果，其中掩码掩盖了20%的比例。

## 三、创新点

### 1. 去除目标检测器
ALBEF不再依赖目标检测器，直接使用Transformer处理图像和文本，降低了计算复杂度并减少了对外部标注数据的依赖。

### 2. 动量编码器（Momentum Encoder）
引入动量编码器（momentum model），通过指数移动平均（EMA）更新参数，起到以下作用：
- 防止训练不稳定：提供更平滑的目标，减少训练波动。
- 提升泛化能力：融合历史信息，类似于集成学习的效果，对抗过拟合。
- 提高对比学习质量：提供更稳定的正样本向量，增强正负样本区分度。
- 在计算ITC（图像-文本对比）学习，使用了动量模型产生的负样本队列，大小4096，ITC的计算量比CLIP小很多。

### 3. 软标签（Soft Target）
动量编码器输出的soft target可以缓解弱监督数据带来的噪声问题，使模型在训练过程中更加稳健。

## 四、训练细节
- 硬件配置：使用8个NVIDIA A100 GPU，batch size为512。
- 优化器：AdamW，权重衰减为0.02。
- 学习率调度：初始1000次迭代线性预热至1e-4，随后按余弦退火warmup至1e-5。
- 数据增强：采用RandAugment（不包含颜色变换），图像分辨率在预训练时为256x256，在微调时提高到384x384。
- 动量参数：动量模型的动量参数设置为0.995，队列大小为65,536。
- 蒸馏权重：在第一个周期内将蒸馏权重α从0线性增加到0.4。

## 五、总结
ALBEF通过简化模型结构、引入动量编码器和软标签机制，显著提升了视觉-语言预训练模型的效率和鲁棒性。这些创新不仅提高了模型的性能，还降低了计算成本，使其更具实用性和可扩展性。